In [1]:
# Regresia Liniara
# g(X) = yCaciula = Sum(i=1,n) (wi - xi) + b
# vectorial:                      W*X    + b,
# unde W e vectorul de ponderi (in R1)


# obs: ec dreptei este y = m*x + b,
# unde b este bias (translatia), iar m este panta (in cazul nostru este vectorul de caracteristici)

# interpolare => un polinom care imparte perfect punctele
# regresia => o dreapta (daca e liniara, daca nu, un polinom => regresie polinomiala) care aproximeaza impartirea

In [2]:
# mse = Mean Squared Error,
# o comparatie intre pretul prezis si pretul real (la testare avem si labelul real sa comparam predictiile noastre)

# Obs: e mai usor la calcul sa folosim (yi - yj)^2 decat radical din asta, care e distanta euclidiana
# MSE(yCaciula, y) = sum(i=1, n) (yi - yj)^2 ; Ala nu cred ca e yj, ci yCaciula


In [3]:
# Bias vs Variance
# varianta de data de zgomotul din datele tale pe care modelul tau le invata prea bine, pe de rost
# biasul e dat de modelul tau prea simpul

# overfitting => un polinom de grad mai mare care aproximeaza foarte bine modelele de antrenare,
# dar pe cazuri generale va da fail
# un model mai bun ar fi unul mai simplu, ca un polinom de gr 2 (in cazul nostru 2D)

# un model f simplu (gen o dreapta) nu aproximeaza bine
# bias f mare => model pre simplu

# la interpolari putem face poilnoame de mai multe grade;
# daca avem un polinom de grad 20, putem sa mai scadem din grade


In [4]:
# Obs: Regresia e pt valori continue; noi pana acum am facut clasificari pe variablie discrete

# formula regresiei liniare:
# sum(i = 1, n) (wi*xi) + b

# Regresia Ridge adauga un factor de regularizare:
# (sum(i = 1, n) (wi*xi) + b) + lambda * norma2(W)
# norma2 (l2) a ponderilor determina magnitudinea ponderilor (diferenta dintre ponderea minima si ponderea maxima)
# vrem magnitudine foarte mica, dar niciodata 0

# Regresia Lasso este la fel ca regresia ridge, dar in loc de norma l2 foloseste norma l1
# norma l1 permite si valori de 0 => caracteristicile irelevante sunt indepartate
# lambda e o constanta determinata experimental care spune cat de importanta este magnitudinea ponderilor

# Obs: regresiile ridge si lasso sunt tot regresii liniare (drepte), dar particulare


In [5]:
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge, Lasso
# from sklearn.linear_model import LinearRegression, RidgeRegression, LassoRegression

modelLinRegr = LinearRegression();

trainData = np.load('data_lab6/data/training_data.npy',allow_pickle = True)
trainLabels = np.load('data_lab6/data/prices.npy',allow_pickle = True)
# testData = np.load('./data_lab6/data/test_sentences.npy',allow_pickle = True)
# testLabels = np.load('./data_lab6/data/test_labels.npy',allow_pickle = True)

# modelLinRegr.fit(trainData, trainLabels);
# y = modelLinRegr.predict(testData)



In [6]:
from sklearn import preprocessing

def normalizer(trainData, testData = None):
    scaler = preprocessing.StandardScaler()
    scaler.fit(trainData)
    scaledXTrain = scaler.transform(trainData)
    if testData is None:
        return scaledXTrain, None
    scaledXTest = scaler.transform(testData)
    return scaledXTrain, scaledXTest

trainData, testData = normalizer(trainData)

In [7]:
# Validarea incrucisata (k fold)
# avem un intreg si k = 3 (3 folduri de facut, trebuie sa impartim in 3 bucati egale)
# la  iteratia 1, primele 2 folduri sunt folosite pt antrenarea modelului si al treilea este folosit la testare)
# la iteratia 2, antrenam 1 si 3 si testam pe 2
# la iteratia 3, folosim 2, 3 pt antrenare si 1 pt testare
# la fiecare obtinem o eroare: e1, e2, e3
# E = e1 + e2 + e3
# formala pt eroare folosita = (yCaciula - yi)^2, unde yCaciula este eticheta reala si yi este eticheta prezisa


In [18]:
def trainTestModel(model, trainData, trainLabels, testData, testLabels):
    model.fit(trainData, trainLabels)
    yPred = model.predict(testData)
    mse = mean_squarred_error(yPred, testLabels)
    mae = mean_absolute_error(yPred, testLabels)
    return mse, mae

def kFold(model, trainData, trainLabels, k = 3):
    samplesPerFold = len(trainData)//k # how many samples we have in every fold
    mse = [0] * k
    mae = [0] * k        
    meanMse = 0
    meanMae = 0
    for i in range(1, k+1):
        crtTrainData = np.empty(shape = (0,1626,14))
        crtTrainLabels = np.empty(shape = (0,1626,14))
        for j in range(1, k+1):
            if j != i:
                lowerIdx = samplesPerFold * (j-1)
                upperIdx = samplesPerFold * j
                crtTrainData = np.concatenate(crtTrainData, trainData[lowerIdx : upperIdx])
                crtTrainLabels = np.concatenate(crtTrainLabels, trainLabels[lowerIdx : upperIdx])
        
        lowerIdx = samplesPerFold * (i-1)
        upperIdx = samplesPerFold * i
        mse[i], mae[i] = trainTestModel(model, crtTrainData, crtTrainLabels, testData[lowerIdx : upperIdx], test[lowerIdx : upperIdx])
        meanMse += mse[i]
        meanMae += mae[i]
    meanMse /= k
    meanMae /= k
    return meanMse, meanMae

meanMse, meanMae = kFold(modelLinRegr, trainData, trainLabels)
print("meanMse = {},  meanMae = {}".format(meanMse, meanMae))
                
    

TypeError: only integer scalar arrays can be converted to a scalar index

In [ ]:
# obs: cel mai semnificativ coeficient este cel cu valoare absoluta cea mai mare:
# wSemnificativ este w pt care avem max(abs(w) for w in W)

# model = Ridge(alpha = 0)
# model.fit(trainData, trainLabels)
# coef = model.coef

# np.argmax(np.abs(coef))
# np.argsort(np.abs(coef))
# np.argsort  returneaza o lista de indici ca si cand val ar fi sortate cresc

